# exp_09 Feature Reduction


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
config = load_experiment_config()
train_df, val_df, _ = data_loading.load_berlin_splits(DATA_DIR / 'phase_2_splits')
feature_cols = data_loading.get_feature_columns(train_df)

x_train = train_df[feature_cols]
x_val = val_df[feature_cols]

x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(x_train, x_val=x_val)

y_train, label_to_idx, _ = preprocessing.encode_genus_labels(train_df['genus_latin'])

model = models.create_model('random_forest')
model.fit(x_train_scaled, y_train)

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

candidate_k = config['setup_ablation']['feature_reduction']['candidate_k']
results = []
for k in candidate_k:
    selected = importance.head(k)['feature'].tolist()
    x_train_k = train_df[selected]
    x_val_k = val_df[selected]
    x_train_k_scaled, x_val_k_scaled, _, _ = preprocessing.scale_features(x_train_k, x_val=x_val_k)
    y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()
    cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])
    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(rf, x_train_k_scaled, y_train, train_df['block_id'].values, cv)
    results.append({'n_features': k, 'val_f1_mean': metrics['val_f1_mean'], 'val_f1_std': metrics['val_f1_std']})

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_09_feature_reduction'
fig_dir.mkdir(parents=True, exist_ok=True)
selected_k = int(results_df.sort_values('val_f1_mean', ascending=False).iloc[0]['n_features'])
visualization.plot_pareto_curve(results_df, selected_k=selected_k, output_path=fig_dir / 'pareto_curve.png')
